# KING2 + LLaMA Factory Fine-tuning
---
**تدريب نماذج KING2 باستخدام LLaMA Factory**

يدعم هذا النوت بوك:
- Qwen3.5-9B (Apache 2.0)
- Gemma 4 E4B (Apache 2.0)
- LoRA / QLoRA
- حفظ لينك Colab مع GitHub

## 1. ربط Google Drive مع GitHub

ارفع مجلد `data/` من مشروع KING2 (فيه `king2_training.json` + `dataset_info.json`)

In [ ]:
# @title 1. تثبيت LLaMA Factory
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Clone LLaMA Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# Install
!pip install -e ".[torch,metrics]"
!pip install unsloth

print('LLaMA Factory ready!')

In [ ]:
# @title 2. رفع بيانات KING2
import os
from google.colab import files

print('ارفع الملفات التالية:')
print('1. king2_training.json')
print('2. dataset_info.json')

uploaded = files.upload()

import json
for fname in uploaded.keys():
    print(f'Loaded: {fname}')

# Copy to LLaMA Factory data dir
!mkdir -p data
!cp *.json data/
!ls -la data/king2_training.json data/dataset_info.json

In [ ]:
# @title 3. اختيار النموذج
MODEL_CHOICE = 'qwen'  # @param ['qwen', 'gemma']

if MODEL_CHOICE == 'qwen':
    model_name = 'Qwen/Qwen3.5-9B'
    template = 'qwen'
    output_dir = 'king2-qwen3.5-9b'
else:
    model_name = 'google/gemma-4-e4b-it'
    template = 'gemma'
    output_dir = 'king2-gemma-4-e4b'

print(f'Model: {model_name}')
print(f'Template: {template}')
print(f'Output: {output_dir}')

In [ ]:
# @title 4. تشغيل التدريب (LoRA + QLoRA)
import yaml

config = {
    # Model
    'model_name_or_path': model_name,
    'template': template,
    'finetuning_type': 'lora',
    
    # Dataset
    'dataset': 'king2_training',
    'dataset_dir': 'data',
    'max_samples': 100000,
    'cutoff_len': 2048,
    
    # LoRA
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'lora_target': 'all',
    
    # Quantization
    'quantization_bit': 4,
    'quantization_method': 'bitsandbytes',
    
    # Training
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 2.0e-4,
    'num_train_epochs': 3.0,
    'max_grad_norm': 1.0,
    'warmup_ratio': 0.1,
    'logging_steps': 5,
    'save_steps': 100,
    'save_total_limit': 2,
    
    # Optimizer
    'optim': 'adamw_torch',
    'lr_scheduler_type': 'cosine',
    
    # Precision
    'fp16': not torch.cuda.is_bf16_supported(),
    'bf16': torch.cuda.is_bf16_supported(),
    
    # Speed
    'flash_attn': 'auto',
    
    # Output
    'output_dir': output_dir,
    'report_to': 'none',
}

with open('train_config.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)

print('Config saved to train_config.yaml')
print('Starting training...')

# Run training
!llamafactory-cli train train_config.yaml

print('Training complete!')

In [ ]:
# @title 5. حفظ النموذج (LoRA adapter)
import shutil

# تحديد مسار أحدث checkpoint
import glob
checkpoints = sorted(glob.glob(f'{output_dir}/checkpoint-*'))
if checkpoints:
    latest = checkpoints[-1]
    print(f'Latest checkpoint: {latest}')
    
    # Create final adapter dir
    final_dir = f'{output_dir}/adapter'
    !cp -r "{latest}"/* {final_dir}/
    
    # Zip for download
    !zip -r "{output_dir}.zip" {output_dir}/
    
    print(f'LoRA adapter saved to: {final_dir}/')
    print(f'Zip: {output_dir}.zip')
    
    # Download
    from google.colab import files
    files.download(f'{output_dir}.zip')
else:
    print('No checkpoints found')

In [ ]:
# @title 6. اختبار النموذج المدرّب
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

# Load the fine-tuned model
args = dict(
    model_name_or_path=model_name,
    adapter_dir=f'{output_dir}/adapter',
    template=template,
    finetuning_type='lora',
    quantization_bit=4,
)

chat_model = ChatModel(args)
print('Model loaded for inference!')

# Test questions
tests = [
    'من أنت؟',
    'ما هو الجمع في الرياضيات؟',
    'اشرح لي نظرية فيثاغورس',
    'اكتب دالة Python لجمع رقمين',
    'ما هو السلام الإيجابي؟',
]

for question in tests:
    print(f'\nQ: {question}')
    messages = []
    messages.append({'role': 'user', 'content': question})
    response = ''
    for new_text in chat_model.stream_chat(messages):
        response += new_text
    print(f'A: {response}')
    print('---')

In [ ]:
# @title 7. (اختياري) تصدير النموذج المدمج
# هذا يدمج LoRA مع النموذج الأصلي ويصدر GGUF للتشغيل المحلي

merge = input('هل تريد دمج LoRA وإصدار GGUF؟ (y/n): ')

if merge.lower() == 'y':
    export_config = {
        'model_name_or_path': model_name,
        'adapter_dir': f'{output_dir}/adapter',
        'template': template,
        'finetuning_type': 'lora',
        'export_dir': f'{output_dir}-merged',
        'export_size': 2,
        'export_device': 'cpu',
        'export_quantization_bit': 4,
        'export_quantization_dataset': 'data/king2_training.json',
        'export_quantization_nsamples': 128,
    }
    
    with open('export_config.yaml', 'w') as f:
        yaml.dump(export_config, f, allow_unicode=True)
    
    !llamafactory-cli export export_config.yaml
    
    # Zip the merged model
    !zip -r "{output_dir}-merged.zip" {output_dir}-merged/
    
    print(f'GGUF exported to: {output_dir}-merged/')
    from google.colab import files
    files.download(f'{output_dir}-merged.zip')
else:
    print('Skipped.')

print('\nDone! استخدم LoRA adapter أو GGUF حسب احتياجك.')